# Mental Health Support Chatbot (Fine-Tuning DistilGPT2)

## Objective
Build a basic chatbot that provides supportive and empathetic responses for stress, anxiety, and emotional wellness using the **EmpatheticDialogues** dataset from Facebook AI.

## Model Base
We will use **DistilGPT2**, a distilled version of GPT-2 that is smaller, faster, and more efficient while retaining most of its capabilities.

In [ ]:
!pip install -r requirements.txt

In [ ]:
import os
import torch
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Dataset Loading and Exploration
The **EmpatheticDialogues** dataset contains conversations where one person is talking about a personal situation and the other person responds with empathy.

In [ ]:
# Load the dataset
dataset = load_dataset("empathetic_dialogues")
print(dataset)

In [ ]:
# Explore a sample
df_train = pd.DataFrame(dataset['train'])
print(df_train.head())

# Visualize emotion labels distribution
plt.figure(figsize=(12, 6))
sns.countplot(data=df_train, x='context', order=df_train['context'].value_counts().index)
plt.xticks(rotation=90)
plt.title("Distribution of Emotions in EmpatheticDialogues")
plt.show()

## 2. Data Preprocessing
We need to format the dialogues into a format suitable for GPT-2 training: `<|user|> prompt <|assistant|> response`.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token

def preprocess_function(examples):
    # Combine situation and utterance into a single prompt-response pair
    # Format it as: User: [situation] Assistant: [utterance]
    texts = [f"User: {p}\nAssistant: {u}{tokenizer.eos_token}" for p, u in zip(examples['prompt'], examples['utterance'])]
    return tokenizer(texts, truncation=True, padding="max_length", max_length=128)

tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=dataset['train'].column_names)

## 3. Model Fine-Tuning
We will use the `Trainer` API to fine-tune DistilGPT2.

In [ ]:
model = AutoModelForCausalLM.from_pretrained("distilgpt2")

training_args = TrainingArguments(
    output_dir="./mental_health_model",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    weight_decay=0.01,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    save_steps=500,
    logging_steps=100,
    push_to_hub=False,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

# trainer.train() # Uncomment to train

## 4. Evaluation and Inference
After training, we can test the model with some prompts.

In [ ]:
def generate_response(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_length=100, num_return_sequences=1, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(generate_response("I feel very stressed about my exams."))